In [ ]:
!kaggle datasets download -d atharvaingle/crop-recommendation-dataset
!kaggle datasets download -d akshatgupta7/crop-yield-in-indian-states-dataset
!kaggle datasets download -d miadul/irrigation-water-requirement-prediction-dataset
!kaggle datasets download -d shankarpriya2913/crop-and-soil-dataset

In [ ]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn

## 1 Crop Recommendation Model

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

### Data Exploration

In [2]:
df = pd.read_csv("./data/crop-recommendation-dataset.csv")

# basic info
print(df.head())
print(df.shape)
print(df.info())
print(df.describe())

    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice
(2200, 8)
<class 'pandas.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   str    
dtypes: float64(4), int64(3), str(1)
memory usage: 137.6 KB
None
              

In [3]:
print(df["label"].unique())

<StringArray>
[       'rice',       'maize',    'chickpea', 'kidneybeans',  'pigeonpeas',
   'mothbeans',    'mungbean',   'blackgram',      'lentil', 'pomegranate',
      'banana',       'mango',      'grapes',  'watermelon',   'muskmelon',
       'apple',      'orange',      'papaya',     'coconut',      'cotton',
        'jute',      'coffee']
Length: 22, dtype: str


In [4]:
print(df.isnull().sum())
print(df.duplicated().sum())

N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64
0


In [5]:
# Encode target
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
print("Classes:", le.classes_)

Classes: ['apple' 'banana' 'blackgram' 'chickpea' 'coconut' 'coffee' 'cotton'
 'grapes' 'jute' 'kidneybeans' 'lentil' 'maize' 'mango' 'mothbeans'
 'mungbean' 'muskmelon' 'orange' 'papaya' 'pigeonpeas' 'pomegranate'
 'rice' 'watermelon']


In [6]:
# Split features and target
X = df.drop(['label', 'label_enc'], axis=1)
y = df['label_enc']

In [7]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(1760, 7) (440, 7)


### Train XGBoost

In [8]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)
model.fit(X_train, y_train)

d:\smart-agriculture\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:56:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [9]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 98.86%
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       0.95      1.00      0.98        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      0.85      0.92        20
       maize       1.00      1.00      1.00        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.95      1.00      0.98        20
    mungbean       0.91      1.00      0.95        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        20
      papa

In [10]:
import os
import joblib

os.makedirs('model', exist_ok=True)

# Save XGBoost native format
model.save_model('model/crop_recommandation.json')

# Save label encoder
joblib.dump(le, 'model/crop_recommandation.pkl')

# Model saved to model/crop_recommandation.json
# Label encoder saved to model/crop_recommandation.pkl

['model/crop_recommandation.pkl']

### make prediction

In [11]:
# Load model and encoder
model = xgb.XGBClassifier()
model.load_model('./model/crop_recommandation.json')
le = joblib.load('./model/crop_recommandation.pkl')

# Single sample prediction (N, P, K, temperature, humidity, ph, rainfall)
sample = np.array([[90, 42, 43, 20.87, 82.00, 6.50, 202.93]])

pred_num = model.predict(sample)
pred_crop = le.inverse_transform(pred_num)

print(f"Predicted crop: {pred_crop[0]}")

Predicted crop: rice


## 2 Agricultural Crop Yield in Indian States 

In [12]:
crop_yield_df = pd.read_csv('./data/crop-yield-in-indian-states-dataset.csv')
print(crop_yield_df.head())
print(crop_yield_df.shape)
print(crop_yield_df.columns.tolist())

           Crop  Crop_Year       Season  State     Area  Production  \
0      Arecanut       1997  Whole Year   Assam  73814.0       56708   
1     Arhar/Tur       1997  Kharif       Assam   6637.0        4685   
2   Castor seed       1997  Kharif       Assam    796.0          22   
3      Coconut        1997  Whole Year   Assam  19656.0   126905000   
4  Cotton(lint)       1997  Kharif       Assam   1739.0         794   

   Annual_Rainfall  Fertilizer  Pesticide        Yield  
0           2051.4  7024878.38   22882.34     0.796087  
1           2051.4   631643.29    2057.47     0.710435  
2           2051.4    75755.32     246.76     0.238333  
3           2051.4  1870661.52    6093.36  5238.051739  
4           2051.4   165500.63     539.09     0.420909  
(19689, 10)
['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield']


In [13]:
yield_model_df = crop_yield_df.drop(columns=['Crop_Year', 'Fertilizer', 'Production']).copy()
yield_categorical_columns = yield_model_df.select_dtypes(include=['object', 'string']).columns.tolist()
yield_model_df = pd.get_dummies(yield_model_df, columns=yield_categorical_columns, drop_first=True)

yield_bool_columns = yield_model_df.select_dtypes(include=['bool']).columns
if len(yield_bool_columns) > 0:
    yield_model_df[yield_bool_columns] = yield_model_df[yield_bool_columns].astype(int)

print(yield_model_df.head())
print(yield_model_df.shape)

      Area  Annual_Rainfall  Pesticide        Yield  Crop_Arhar/Tur  \
0  73814.0           2051.4   22882.34     0.796087               0   
1   6637.0           2051.4    2057.47     0.710435               1   
2    796.0           2051.4     246.76     0.238333               0   
3  19656.0           2051.4    6093.36  5238.051739               0   
4   1739.0           2051.4     539.09     0.420909               0   

   Crop_Bajra  Crop_Banana  Crop_Barley  Crop_Black pepper  Crop_Cardamom  \
0           0            0            0                  0              0   
1           0            0            0                  0              0   
2           0            0            0                  0              0   
3           0            0            0                  0              0   
4           0            0            0                  0              0   

   ...  State_Odisha  State_Puducherry  State_Punjab  State_Sikkim  \
0  ...             0                 0  

In [14]:
yield_X = yield_model_df.drop(columns=['Yield'])
yield_y = yield_model_df['Yield']

yield_X_train, yield_X_test, yield_y_train, yield_y_test = train_test_split(
    yield_X, yield_y, test_size=0.2, random_state=42
)

print(yield_X_train.shape, yield_X_test.shape)

(15751, 91) (3938, 91)


In [15]:
yield_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
)

yield_model.fit(
    yield_X_train,
    yield_y_train,
    eval_set=[(yield_X_test, yield_y_test)],
    verbose=100
)

[0]	validation_0-rmse:856.64463
[100]	validation_0-rmse:227.16989
[200]	validation_0-rmse:220.91889
[300]	validation_0-rmse:221.78420
[400]	validation_0-rmse:222.02304
[499]	validation_0-rmse:221.99544


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [16]:
yield_y_pred = yield_model.predict(yield_X_test)
yield_rmse = np.sqrt(mean_squared_error(yield_y_test, yield_y_pred))
yield_mae = mean_absolute_error(yield_y_test, yield_y_pred)
yield_r2 = r2_score(yield_y_test, yield_y_pred)

print(f'RMSE: {yield_rmse:.4f}')
print(f'MAE: {yield_mae:.4f}')
print(f'R2: {yield_r2:.4f}')

RMSE: 221.9954
MAE: 15.9269
R2: 0.9385


In [17]:
import os
import pickle

os.makedirs('./model', exist_ok=True)
yield_model.save_model('./model/crop_yield.json')

with open('./model/crop_yield.pkl', 'wb') as f:
    pickle.dump({
        'feature_columns': yield_X.columns.tolist(),
        'dropped_columns': ['Crop_Year', 'Fertilizer', 'Production'],
        'target_column': 'Yield'
    }, f)

print('Saved model to ./model/crop_yield.json')
print('Saved artifacts to ./model/crop_yield.pkl')

Saved model to ./model/crop_yield.json
Saved artifacts to ./model/crop_yield.pkl


In [18]:
import pickle

loaded_yield_model = xgb.XGBRegressor()
loaded_yield_model.load_model('./model/crop_yield.json')

with open('./model/crop_yield.pkl', 'rb') as f:
    yield_artifacts = pickle.load(f)

sample_row = crop_yield_df.iloc[[0]].copy()
actual_yield = sample_row['Yield'].iloc[0]
sample_row = sample_row.drop(columns=yield_artifacts['dropped_columns'] + [yield_artifacts['target_column']])
sample_row = pd.get_dummies(sample_row)
sample_row = sample_row.reindex(columns=yield_artifacts['feature_columns'], fill_value=0)

predicted_yield = loaded_yield_model.predict(sample_row)[0]
print(f'Actual yield: {actual_yield:.4f}')
print(f'Predicted yield from saved model: {predicted_yield:.4f}')

Actual yield: 0.7961
Predicted yield from saved model: 7.2213


## 3 irrigation requirement prediction

In [19]:
irrigation_df = pd.read_csv('./data/irrigation-requirement-prediction-dataset.csv')
print(irrigation_df.head())

  Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  \
0      Clay     6.14          36.48            0.42                     2.17   
1      Silt     6.41          50.56            0.38                     0.23   
2     Sandy     7.71          40.07            1.09                     2.18   
3      Clay     5.96          12.75            1.56                     0.40   
4      Clay     7.76          18.58            0.95                     2.52   

   Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh  \
0          21.90     31.19      1167.70            4.01            1.97   
1          36.50     26.01       831.28           10.72           16.82   
2          41.83     76.41      1844.45            7.75           19.03   
3          37.22     43.32       306.26            8.90           11.44   
4          22.38     86.44      1875.63           10.39           11.26   

  Crop_Type Crop_Growth_Stage  Season Irrigation_Type Water_Source  

In [20]:
columns_to_drop = [
    'Crop_Growth_Stage',
    'Irrigation_Type',
    'Sunlight_Hours',
    'Water_Source',
    'Mulching_Used',
    'Organic_Carbon',
    'Electrical_Conductivity',
    'Field_Area_hectare',
    'Previous_Irrigation_mm',
    'Crop_Type'
]

irrigation_model_df = irrigation_df.drop(columns=columns_to_drop).copy()
print(irrigation_model_df.head())
print(irrigation_model_df.columns.tolist())

  Soil_Type  Soil_pH  Soil_Moisture  Temperature_C  Humidity  Rainfall_mm  \
0      Clay     6.14          36.48          21.90     31.19      1167.70   
1      Silt     6.41          50.56          36.50     26.01       831.28   
2     Sandy     7.71          40.07          41.83     76.41      1844.45   
3      Clay     5.96          12.75          37.22     43.32       306.26   
4      Clay     7.76          18.58          22.38     86.44      1875.63   

   Wind_Speed_kmh  Season   Region Irrigation_Need  
0            1.97    Rabi    South             Low  
1           16.82    Zaid  Central          Medium  
2           19.03    Rabi    South             Low  
3           11.44  Kharif    North          Medium  
4           11.26    Zaid    South          Medium  
['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Wind_Speed_kmh', 'Season', 'Region', 'Irrigation_Need']


In [21]:
feature_label_encoders = {}
categorical_columns = irrigation_model_df.select_dtypes(include=['object', 'string']).columns.tolist()
categorical_columns.remove('Irrigation_Need')

for column in categorical_columns:
    encoder = LabelEncoder()
    irrigation_model_df[column] = encoder.fit_transform(irrigation_model_df[column])
    feature_label_encoders[column] = encoder

target_encoder = LabelEncoder()
irrigation_model_df['Irrigation_Need_Enc'] = target_encoder.fit_transform(irrigation_model_df['Irrigation_Need'])

print('Feature encoders:', list(feature_label_encoders.keys()))
print('Target classes:', target_encoder.classes_)

Feature encoders: ['Soil_Type', 'Season', 'Region']
Target classes: ['High' 'Low' 'Medium']


In [22]:
X = irrigation_model_df.drop(columns=['Irrigation_Need', 'Irrigation_Need_Enc'])
y = irrigation_model_df['Irrigation_Need_Enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(8000, 9) (2000, 9)


In [23]:
irrigation_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(target_encoder.classes_),
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

irrigation_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [24]:
y_pred = irrigation_model.predict(X_test)
y_pred_labels = target_encoder.inverse_transform(y_pred)
y_test_labels = target_encoder.inverse_transform(y_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test_labels, y_pred_labels))

Accuracy: 0.737
              precision    recall  f1-score   support

        High       0.59      0.33      0.42        67
         Low       0.79      0.82      0.80      1173
      Medium       0.66      0.65      0.65       760

    accuracy                           0.74      2000
   macro avg       0.68      0.60      0.63      2000
weighted avg       0.73      0.74      0.73      2000



In [25]:
import os
import pickle

os.makedirs('./model', exist_ok=True)
irrigation_model.save_model('./model/irrigation_need.json')

with open('./model/irrigation_need.pkl', 'wb') as f:
    pickle.dump({
        'feature_label_encoders': feature_label_encoders,
        'target_encoder': target_encoder,
        'feature_columns': X.columns.tolist()
    }, f)

print('Saved model to ./model/irrigation_need.json')
print('Saved encoders to ./model/irrigation_need.pkl')

Saved model to ./model/irrigation_need.json
Saved encoders to ./model/irrigation_need.pkl


In [26]:
print(irrigation_df.iloc[1])

Soil_Type                         Silt
Soil_pH                           6.41
Soil_Moisture                    50.56
Organic_Carbon                    0.38
Electrical_Conductivity           0.23
Temperature_C                     36.5
Humidity                         26.01
Rainfall_mm                     831.28
Sunlight_Hours                   10.72
Wind_Speed_kmh                   16.82
Crop_Type                        Maize
Crop_Growth_Stage            Flowering
Season                            Zaid
Irrigation_Type                  Canal
Water_Source               Groundwater
Field_Area_hectare               12.22
Mulching_Used                      Yes
Previous_Irrigation_mm           33.56
Region                         Central
Irrigation_Need                 Medium
Name: 1, dtype: object


In [27]:
# Load irrigation model and encoders
import pickle

loaded_irrigation_model = xgb.XGBClassifier()
loaded_irrigation_model.load_model('./model/irrigation_need.json')

with open('./model/irrigation_need.pkl', 'rb') as f:
    irrigation_artifacts = pickle.load(f)

loaded_feature_encoders = irrigation_artifacts['feature_label_encoders']
loaded_target_encoder = irrigation_artifacts['target_encoder']
loaded_feature_columns = irrigation_artifacts['feature_columns']

# Single sample prediction
sample_input = pd.DataFrame([{
    'Soil_Type': 'Silt',
    'Soil_pH': 6.41,
    'Soil_Moisture': 50.56,
    'Temperature_C': 36.5,
    'Humidity': 26.01,
    'Rainfall_mm': 831.28,
    'Sunlight_Hours': 10.72,
    'Wind_Speed_kmh': 16.82,
    'Season': 'Zaid',
    'Region': 'Central'
}])

for column, encoder in loaded_feature_encoders.items():
    sample_input[column] = encoder.transform(sample_input[column])

sample_input = sample_input[loaded_feature_columns]
pred_num = loaded_irrigation_model.predict(sample_input)
pred_label = loaded_target_encoder.inverse_transform(pred_num)

print(f'Predicted irrigation need: {pred_label[0]}')

Predicted irrigation need: Medium


## 4 Fertilizer Recommendation

In [28]:
FERTILIZER_DATA_PATH = './data/fertilizer-prediction.csv'
FERTILIZER_RANDOM_STATE = 42

fertilizer_df = pd.read_csv(FERTILIZER_DATA_PATH)
print('Data loaded:', fertilizer_df.shape)
print(fertilizer_df.head())
print('\nColumn dtypes:\n', fertilizer_df.dtypes)
print('\nNull values:\n', fertilizer_df.isnull().sum())

Data loaded: (100000, 9)
   Temparature  Humidity  Moisture Soil Type    Crop Type  Nitrogen  \
0           32        51        41       Red  Ground Nuts         7   
1           35        58        35     Black       Cotton         4   
2           27        55        43     Sandy    Sugarcane        28   
3           33        56        56     Loamy  Ground Nuts        37   
4           32        70        60       Red  Ground Nuts         4   

   Potassium  Phosphorous Fertilizer Name  
0          3           19        14-35-14  
1         14           16            Urea  
2          0           17           20-20  
3          5           24           28-28  
4          6            9        14-35-14  

Column dtypes:
 Temparature        int64
Humidity           int64
Moisture           int64
Soil Type            str
Crop Type            str
Nitrogen           int64
Potassium          int64
Phosphorous        int64
Fertilizer Name      str
dtype: object

Null values:
 Temparature  

In [29]:
FERTILIZER_CATEGORICAL_COLS = ['Soil Type', 'Crop Type']
FERTILIZER_TARGET_COL = 'Fertilizer Name'

fertilizer_label_encoders = {}

for col in FERTILIZER_CATEGORICAL_COLS:
    le = LabelEncoder()
    fertilizer_df[col] = le.fit_transform(fertilizer_df[col])
    fertilizer_label_encoders[col] = le
    print(f"Encoded '{col}': {list(le.classes_)}")

fertilizer_target_le = LabelEncoder()
fertilizer_df[FERTILIZER_TARGET_COL] = fertilizer_target_le.fit_transform(fertilizer_df[FERTILIZER_TARGET_COL])
fertilizer_label_encoders[FERTILIZER_TARGET_COL] = fertilizer_target_le
print(f"\nTarget classes ({len(fertilizer_target_le.classes_)}):\n  {list(fertilizer_target_le.classes_)}")

Encoded 'Soil Type': ['Black', 'Clayey', 'Loamy', 'Red', 'Sandy']
Encoded 'Crop Type': ['Barley', 'Cotton', 'Ground Nuts', 'Maize', 'Millets', 'Oil seeds', 'Paddy', 'Pulses', 'Sugarcane', 'Tobacco', 'Wheat']

Target classes (7):
  ['10-26-26', '14-35-14', '17-17-17', '20-20', '28-28', 'DAP', 'Urea']


In [30]:
FERTILIZER_FEATURE_COLS = [c for c in fertilizer_df.columns if c != FERTILIZER_TARGET_COL]
fertilizer_X = fertilizer_df[FERTILIZER_FEATURE_COLS]
fertilizer_y = fertilizer_df[FERTILIZER_TARGET_COL]

print(f"\nFeatures: {FERTILIZER_FEATURE_COLS}")
print(f"X shape: {fertilizer_X.shape} | y shape: {fertilizer_y.shape}")

fertilizer_X_train, fertilizer_X_test, fertilizer_y_train, fertilizer_y_test = train_test_split(
    fertilizer_X, fertilizer_y, test_size=0.2, random_state=FERTILIZER_RANDOM_STATE, stratify=fertilizer_y
)

print(f"\nTrain size: {len(fertilizer_X_train)} | Test size: {len(fertilizer_X_test)}")


Features: ['Temparature', 'Humidity', 'Moisture', 'Soil Type', 'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorous']
X shape: (100000, 8) | y shape: (100000,)

Train size: 80000 | Test size: 20000


In [31]:
fertilizer_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=FERTILIZER_RANDOM_STATE,
    n_jobs=-1,
)

print('\nTraining XGBoost...')
fertilizer_model.fit(
    fertilizer_X_train,
    fertilizer_y_train,
    eval_set=[(fertilizer_X_test, fertilizer_y_test)],
    verbose=50,
)


Training XGBoost...
[0]	validation_0-mlogloss:1.94582


d:\smart-agriculture\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:57:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[50]	validation_0-mlogloss:1.94999
[100]	validation_0-mlogloss:1.95515
[150]	validation_0-mlogloss:1.96165
[200]	validation_0-mlogloss:1.96761
[250]	validation_0-mlogloss:1.97378
[299]	validation_0-mlogloss:1.97936


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [32]:
fertilizer_y_pred = fertilizer_model.predict(fertilizer_X_test)
fertilizer_acc = accuracy_score(fertilizer_y_test, fertilizer_y_pred)
print(f"\nTest Accuracy: {fertilizer_acc * 100:.2f}%")

fertilizer_y_test_labels = fertilizer_target_le.inverse_transform(fertilizer_y_test)
fertilizer_y_pred_labels = fertilizer_target_le.inverse_transform(fertilizer_y_pred)
print('\nClassification Report:')
print(classification_report(fertilizer_y_test_labels, fertilizer_y_pred_labels))


Test Accuracy: 14.38%

Classification Report:
              precision    recall  f1-score   support

    10-26-26       0.13      0.13      0.13      2876
    14-35-14       0.15      0.16      0.16      2898
    17-17-17       0.15      0.15      0.15      2834
       20-20       0.15      0.14      0.14      2836
       28-28       0.15      0.14      0.14      2847
         DAP       0.14      0.14      0.14      2844
        Urea       0.14      0.14      0.14      2865

    accuracy                           0.14     20000
   macro avg       0.14      0.14      0.14     20000
weighted avg       0.14      0.14      0.14     20000



In [33]:
fertilizer_fi = pd.Series(
    fertilizer_model.feature_importances_, index=FERTILIZER_FEATURE_COLS
).sort_values(ascending=False)
print('\nFeature Importances:')
print(fertilizer_fi.to_string())


Feature Importances:
Phosphorous    0.127774
Nitrogen       0.127283
Moisture       0.126081
Humidity       0.125677
Temparature    0.124643
Potassium      0.124291
Crop Type      0.123648
Soil Type      0.120604


In [34]:
import os
import pickle

os.makedirs('./model', exist_ok=True)
fertilizer_model.save_model('./model/fertilizer_recommendation.json')

with open('./model/fertilizer_recommendation.pkl', 'wb') as f:
    pickle.dump({
        'label_encoders': fertilizer_label_encoders,
        'feature_columns': FERTILIZER_FEATURE_COLS
    }, f)

print('Saved model to ./model/fertilizer_recommendation.json')
print('Saved encoders to ./model/fertilizer_recommendation.pkl')

Saved model to ./model/fertilizer_recommendation.json
Saved encoders to ./model/fertilizer_recommendation.pkl


In [35]:
import pickle

loaded_fertilizer_model = xgb.XGBClassifier()
loaded_fertilizer_model.load_model('./model/fertilizer_recommendation.json')

with open('./model/fertilizer_recommendation.pkl', 'rb') as f:
    fertilizer_artifacts = pickle.load(f)

loaded_label_encoders = fertilizer_artifacts['label_encoders']
loaded_feature_columns = fertilizer_artifacts['feature_columns']

def predict_fertilizer(sample: dict) -> str:
    row = sample.copy()
    for col in FERTILIZER_CATEGORICAL_COLS:
        row[col] = loaded_label_encoders[col].transform([row[col]])[0]

    input_df = pd.DataFrame([row])[loaded_feature_columns]
    pred_encoded = loaded_fertilizer_model.predict(input_df)[0]
    return loaded_label_encoders[FERTILIZER_TARGET_COL].inverse_transform([pred_encoded])[0]

fertilizer_sample = {
    "Temparature": 33,
    "Humidity": 56,
    "Moisture": 56,
    "Soil Type": "Loamy",
    "Crop Type": "Ground Nuts",
    "Nitrogen": 37,
    "Potassium": 5,
    "Phosphorous": 24,
}

fertilizer_result = predict_fertilizer(fertilizer_sample)
print('\nDemo Prediction from saved model:')
for k, v in fertilizer_sample.items():
    print(f'   {k}: {v}')
print(f'   Recommended Fertilizer: {fertilizer_result}')


Demo Prediction from saved model:
   Temparature: 33
   Humidity: 56
   Moisture: 56
   Soil Type: Loamy
   Crop Type: Ground Nuts
   Nitrogen: 37
   Potassium: 5
   Phosphorous: 24
   Recommended Fertilizer: 28-28
